In [3]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB7
from tensorflow.keras import layers, models, optimizers, metrics, backend as K
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:

# **Set Dataset Path**
dataset_path = "E:\Projects\Strabismus\Denoised"

# **Check if Dataset Exists**
if not os.path.exists(dataset_path):
    raise ValueError(f"Dataset path '{dataset_path}' does not exist. Check your Drive location.")


In [5]:
# Get the number of classes dynamically
class_folders = [folder for folder in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, folder))]
NUM_CLASSES = len(class_folders)

print(f"Detected classes: {class_folders}")
print(f"Number of classes: {NUM_CLASSES}")


Detected classes: ['ESOTROPIA', 'EXOTROPIA', 'HYPERTROPIA', 'HYPOTROPIA', 'NORMAL']
Number of classes: 5


In [6]:
# Set constants
IMG_SIZE = 224  # EfficientNetB7 requires large images
BATCH_SIZE = 8
EPOCHS = 30

# Define ImageDataGenerator with proper split
datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.3
)  # 30% split for validation & test

# Training data (70%)
train_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

# Validation data (30% split into 15% validation & 15% test)
val_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# ✅ **Create Separate Test Set**
test_datagen = ImageDataGenerator(rescale=1.0/255)
test_generator = test_datagen.flow_from_directory(
    dataset_path, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False  # Don't shuffle to keep true labels aligned
)


Found 3619 images belonging to 5 classes.
Found 1551 images belonging to 5 classes.
Found 5170 images belonging to 5 classes.


In [7]:
# ✅ **Class Weights for Imbalance**
y_train = train_generator.classes
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))
print("Class Weights:", class_weights_dict)


  # **Enable Mixed Precision for Memory Optimization**
tf.keras.mixed_precision.set_global_policy('mixed_float16')

Class Weights: {0: 1.034, 1: 0.9942307692307693, 2: 1.0137254901960784, 3: 1.0237623762376238, 4: 0.94}
INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU, compute capability 8.6


In [8]:
# Custom F1-Score Metric Function
class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name="f1_score", **kwargs):
        super(F1Score, self).__init__(name=name, **kwargs)
        self.precision = metrics.Precision()
        self.recall = metrics.Recall()

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.argmax(y_pred, axis=1)
        y_true = tf.argmax(y_true, axis=1)
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)

    def result(self):
        precision = self.precision.result()
        recall = self.recall.result()
        return 2 * ((precision * recall) / (precision + recall + K.epsilon()))

    def reset_state(self):
        self.precision.reset_state()
        self.recall.reset_state()



# Load Pretrained EfficientNetB7 (Frozen)
base_model = EfficientNetB7(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
# **Unfreeze Last Few Layers for Fine-Tuning**
for layer in base_model.layers[-50:]:  # Unfreezing last 20 layers
    layer.trainable = True

# Create Model Architecture
x = base_model.output
x = layers.Conv2D(512, (3, 3), padding='same', activation='relu')(x)  # Extra Feature Extraction
x = layers.BatchNormalization()(x)  # Stabilizes Training
x = layers.GlobalAveragePooling2D()(x)  # Reduces Dimensionality
x = layers.Dense(1024, activation='relu')(x)  # Strong Feature Learning
x = layers.Dropout(0.6)(x)  # Prevent Overfitting
x = layers.Dense(512, activation='relu')(x)  # Deeper Feature Representation
x = layers.LayerNormalization()(x)  # Prevents Feature Scaling Issues
x = layers.Dense(256, activation='relu')(x)  # More Feature Refinement
x = layers.Dropout(0.3)(x)  # Extra Regularization
output = layers.Dense(NUM_CLASSES, activation='softmax')(x)  # Final Classification

# Define Final Model
model = models.Model(inputs=base_model.input, outputs=output)



In [9]:
import tensorflow as tf
from tensorflow.keras import optimizers, metrics

# ✅ Learning Rate Scheduler (Cosine Decay)
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-4, first_decay_steps=10, t_mul=2.0, m_mul=0.9, alpha=1e-6
)

# ✅ Custom AdamW Optimizer (with weight decay inside apply_gradients)
class AdamW(optimizers.Adam):
    def __init__(self, learning_rate=1e-4, weight_decay=1e-5, **kwargs):
        super().__init__(learning_rate=learning_rate, **kwargs)
        self.weight_decay = weight_decay

    def _resource_apply_dense(self, grad, var, apply_state=None):
        if grad is not None and 'kernel' in var.name:  # Apply weight decay only to kernel weights
            grad = grad + self.weight_decay * var
        return super()._resource_apply_dense(grad, var, apply_state)

# ✅ Apply Optimizer
optimizer = AdamW(learning_rate=1e-4, weight_decay=1e-5)


# ✅ Learning Rate Reduction on Plateau
lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# ✅ Early Stopping for Stability
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# ✅ Compile Model
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy', metrics.Precision(name='precision'), metrics.Recall(name='recall')]
)




# Print Model Summary
model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 rescaling (Rescaling)          (None, 224, 224, 3)  0           ['input_1[0][0]']                
                                                                                                  
 normalization (Normalization)  (None, 224, 224, 3)  7           ['rescaling[0][0]']              
                                                                                                  
 rescaling_1 (Rescaling)        (None, 224, 224, 3)  0           ['normalization[0][0]']      

In [ ]:
# Train the Model
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator),
    class_weight=class_weights_dict,# **Balances classes**
    callbacks=[lr_scheduler, early_stopping],
    verbose=1
)


Epoch 1/30
453/453 [==============================] - 627s 1s/step - loss: 1.1168 - accuracy: 0.5800 - precision: 0.6870 - recall: 0.4852 - val_loss: 2.5393 - val_accuracy: 0.3069 - val_precision: 0.3170 - val_recall: 0.2876 - lr: 1.0000e-04
Epoch 2/30
252/453 [===============>..............] - ETA: 3:02 - loss: 0.4419 - accuracy: 0.8518 - precision: 0.8715 - recall: 0.8260

In [ ]:
# 🚀 **Evaluate on Test Set**
test_loss, test_accuracy, test_precision, test_recall, test_f1 = model.evaluate(test_generator)

print(f"\n🚀 Test Set Evaluation:")
print(f"✅ Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"✅ Test Precision: {test_precision:.4f}")
print(f"✅ Test Recall: {test_recall:.4f}")
print(f"✅ Test F1 Score: {test_f1:.4f}")

# 🚀 **Get Predictions**
y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = test_generator.classes

# 🚀 **Classification Report**
print("\n🚀 Classification Report (Test Set):")
print(classification_report(y_true, y_pred_classes))

# 🚀 **Confusion Matrix**
conf_matrix = confusion_matrix(y_true, y_pred_classes)

# **Plot Confusion Matrix**
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Test Set)")
plt.show()
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test Accuracy: {test_accuracy:.4f}")
